# New tree

Having come across a homologous protein in *Escherichia albertii*, I decided I should make a more representative tree that contains more distant homologues.

A BLASTp search in the *nr* database yielded hits with corresponding *E. coli* serotypes, and hits on *E. albertii*, so I will use this to make the tree going forwards.

First, I will use CD-HIT to group related proteins, as the BLAST output contains 5000 hits.

In [30]:
# CD-HIT at 95% identity

!cd-hit -i ../../new_tree_wd/blast/seqdump_nr.fasta -o ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta -c 0.99 -n 5
!cd-hit -i ../../new_tree_wd/blast/seqdump_nr.fasta -o ../../new_tree_wd/CD-HIT/seqdump.cdhit95.fasta -c 0.95 -n 5

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:58:32
Command: cd-hit -i ../../new_tree_wd/blast/seqdump_nr.fasta -o
         ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta -c 0.99
         -n 5

Started: Tue Nov 11 15:47:15 2025
                            Output                              
----------------------------------------------------------------
total seq: 5000
longest and shortest : 543 and 41
Total letters: 1687480
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 2M
Buffer          : 1 X 16M = 16M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 83M

Table limit with the given memory limit:
Max number of representatives: 1380802
Max number of word counting entries: 89500670

comparing sequences from          0  to       5000
.....
     5000  finished        303  clusters

Approximated maximum memory consumption: 84M
writing new database
writing clustering information
program completed !

Total CPU time 0.30
Pr

In [31]:
# Seeing how many clusters are created - trying to make a manageable dataset
!grep -c '^>Cluster' ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta.clstr
!grep -c '^>Cluster' ../../new_tree_wd/CD-HIT/seqdump.cdhit95.fasta.clstr

303
54


In [32]:
!mafft --auto ../../new_tree_wd/CD-HIT/seqdump.cdhit99.fasta > ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.fasta
!mafft --auto ../../new_tree_wd/CD-HIT/seqdump.cdhit95.fasta > ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.fasta

nthread = 0
nthreadpair = 0
nthreadtb = 0
ppenalty_ex = 0
stacksize: 8176 kb
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..

There are 13 ambiguous characters.
  301 / 303
done.

Constructing a UPGMA tree (efffree=0) ... 
  300 / 303
done.

Progressive alignment 1/2... 
STEP   157 / 302 
Reallocating..done. *alloclen = 2088
STEP   302 / 302  h
done.

Making a distance matrix from msa.. 
  300 / 303
done.

Constructing a UPGMA tree (efffree=1) ... 
  300 / 303
done.

Progressive alignment 2/2... 
STEP   175 / 302 
Reallocating..done. *alloclen = 2095
STEP   302 / 302  h
done.

disttbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)

distout=h
rescale = 1
dndpre (aa) Version 7.526
alg=X, model=BLOSUM62, 1.53, +0.12, -0.00, noshift, amax=0.0
0 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 2
sueff_global = 0.100000
nadd = 2
res

Now I will look in Jalview, and use trimAl to remove incomplete/gappy sequences

There are a lot of truncated sequences outwith *E. coli* and *E. albertii* that I aim to remove. However there are a few sequences outwith these groups that are more complete that I want to retain.

Trimal wasn't working properly due to sequences containing ambiguous amino acids (Z and J), so I replaced them with X

In [36]:
!sed -E 's/[ZBJ]/X/g' ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.fasta | sed 's/\*//g' > ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.clean.fasta
!sed -E 's/[ZBJ]/X/g' ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.fasta | sed 's/\*//g' > ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.clean.fasta

In [1]:
!trimal -in ../../new_tree_wd/mafft/seqdump.cdhit95.mafft.clean.fasta -out ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta -resoverlap 0.5 -seqoverlap 50 
!trimal -in ../../new_tree_wd/mafft/seqdump.cdhit99.mafft.clean.fasta -out ../../new_tree_wd/trimal/seqdump.cdhit99.trimal.fasta -resoverlap 0.5 -seqoverlap 50 

In [4]:
!trimal -in ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta -out ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta -gappyout -htmlout ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta.html
!trimal -in ../../new_tree_wd/trimal/seqdump.cdhit99.trimal.fasta -out ../../new_tree_wd/trimal/seqdump.cdhit99.trimalgo.fasta -gappyout -htmlout ../../new_tree_wd/trimal/seqdump.cdhit95.trimal.fasta.html


Automated1 does not sufficiently clean the data, so I will have to set the -seqoverlap and -resoverlap manually.

I used -resoverlap 0.5 and -seqoverlap 50, followed by gappyout. I found that -keepheader resulted in some sequence IDs changing, so I removed that.

For the 99% CD-HIT I also had to manually remove a number of truncated sequences, as I didn't want trimAl to remove sequences with good coverage.

Now time for RAxML-NG

In [50]:
# Best tree
!raxml-ng --search --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --prefix ../../new_tree_wd/raxml/new_tree --threads auto


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 11-Nov-2025 16:39:20 as follows:

raxml-ng --search --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --prefix ../../new_tree_wd/raxml/new_tree --threads auto

Analysis options:
  run mode: ML tree search
  start tree(s): random (10) + parsimony (10)
  random seed: 1762879160
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 10.000000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.000000
  fast CLV updates: ON
  branch l

In [55]:
!raxml-ng --bootstrap --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --bs-trees 100 --prefix ../../new_tree_wd/raxml/new_tree_bs --threads auto


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 11-Nov-2025 16:56:29 as follows:

raxml-ng --bootstrap --msa ../../new_tree_wd/trimal/seqdump.cdhit95.trimalgo.fasta --model BLOSUM62+G4 --bs-trees 100 --prefix ../../new_tree_wd/raxml/new_tree_bs --threads auto

Analysis options:
  run mode: Bootstrapping
  start tree(s): 
  bootstrap replicates: parsimony (100)
  random seed: 1762880189
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 0.100000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.000000
  

In [59]:
!raxml-ng --support --tree ../../new_tree_wd/raxml/new_tree.raxml.bestTree --bs-trees ../../new_tree_wd/raxml/new_tree_bs.raxml.bootstraps --prefix ../../new_tree_wd/raxml/new_tree_support


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 11-Nov-2025 17:08:46 as follows:

raxml-ng --support --tree ../../new_tree_wd/raxml/new_tree.raxml.bestTree --bs-trees ../../new_tree_wd/raxml/new_tree_bs.raxml.bootstraps --prefix ../../new_tree_wd/raxml/new_tree_support

Analysis options:
  run mode: Compute bipartition support (Felsenstein Bootstrap)
  start tree(s): user
  random seed: 1762880926
  SIMD kernels: SSE3
  parallelization: coarse-grained (auto), PTHREADS (auto)

Reading reference tree from file: ../../new_tree_wd/raxml/new_tree.raxml.bestTree
Reference tree size: 30

Reading

And for the 99% ID clusteres

In [1]:
!raxml-ng --search --msa ../../new_tree_wd/trimal/seqdump.cdhit99.trimalgo.fasta --model BLOSUM62+G4 --prefix ../../new_tree_wd/raxml/99_id/new_tree99 --threads auto


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 13-Nov-2025 18:50:54 as follows:

raxml-ng --search --msa ../../new_tree_wd/trimal/seqdump.cdhit99.trimalgo.fasta --model BLOSUM62+G4 --prefix ../../new_tree_wd/raxml/99_id/new_tree99 --threads auto

Analysis options:
  run mode: ML tree search
  start tree(s): random (10) + parsimony (10)
  random seed: 1763059854
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 10.000000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.000000
  fast CLV updates: ON
  

In [3]:
!raxml-ng --bootstrap --msa ../../new_tree_wd/trimal/seqdump.cdhit99.trimalgo.fasta --model BLOSUM62+G4 --bs-trees 100 --prefix ../../new_tree_wd/raxml/99_id/new_tree99_bs --threads 8


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 13-Nov-2025 20:03:55 as follows:

raxml-ng --bootstrap --msa ../../new_tree_wd/trimal/seqdump.cdhit99.trimalgo.fasta --model BLOSUM62+G4 --bs-trees 100 --prefix ../../new_tree_wd/raxml/99_id/new_tree99_bs --threads 8

Analysis options:
  run mode: Bootstrapping
  start tree(s): 
  bootstrap replicates: parsimony (100)
  random seed: 1763064235
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 0.100000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.0000

In [4]:
!raxml-ng --support --tree ../../new_tree_wd/raxml/99_id/new_tree99.raxml.bestTree --bs-trees ../../new_tree_wd/raxml/99_id/new_tree99_bs.raxml.bootstraps --prefix ../../new_tree_wd/raxml/99_id/new_tree99_support


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 14-Nov-2025 10:27:28 as follows:

raxml-ng --support --tree ../../new_tree_wd/raxml/99_id/new_tree99.raxml.bestTree --bs-trees ../../new_tree_wd/raxml/99_id/new_tree99_bs.raxml.bootstraps --prefix ../../new_tree_wd/raxml/new_tree99_support

Analysis options:
  run mode: Compute bipartition support (Felsenstein Bootstrap)
  start tree(s): user
  random seed: 1763116048
  SIMD kernels: SSE3
  parallelization: coarse-grained (auto), PTHREADS (auto)

Reading reference tree from file: ../../new_tree_wd/raxml/99_id/new_tree99.raxml.bestTree
Refere

# Addition type information

I have now generated a tree at 95 and 99% identity clusters. Next, I want to extract the protein IDs and include type information labels of the organism the protein came from.

I downloaded the node labels from the tree on iTOL

In [10]:
grep -oE "[A-Za-z0-9_]+\.[0-9]+" ../../new_tree_wd/type_data/95_node_labels.txt | grep -vE "^[0-9]+\.[0-9]+$" | sort -u

CAD5756258.1
EFN7370458.1
ELD0454108.1
ELT8056872.1
ENT7075801.1
HCX9557601.1
HDT0929104.1
HFV8724395.1
HXE6260377.1
HXK9204701.1
HXX1443132.1
HXX1465758.1
MCS1346834.1
MED9390755.1
MGG7294704.1
ONG07614.1
VCY79869.1
WP_001601761.1
WP_063123747.1
WP_086217519.1
WP_097749723.1
WP_115738912.1
WP_125317197.1
WP_160450279.1
WP_171459728.1
WP_183083471.1
WP_250203824.1
WP_257606670.1
WP_306294993.1
WP_426849671.1


I used ChatGPT to write a script that takes the list of these protein IDs, queries NCBI and outputs a CSV with: protein IDs, organism, strain/isolate and the genome accession.



In [14]:
!python ../../scripts/fetch_protein_metadata.py ../../new_tree_wd/type_data/protein_ids.txt ../../new_tree_wd/type_data/protein_metadata.csv

Loaded 30 protein IDs from ../../new_tree_wd/type_data/protein_ids.txt
 ERROR (; No genome accession found)
[2/30] Fetching EFN7370458 ... ERROR (; No genome accession found)
[3/30] Fetching ELD0454108 ... ERROR (; No genome accession found)
[4/30] Fetching ELT8056872 ... ERROR (; No genome accession found)
[5/30] Fetching ENT7075801 ... ERROR (; No genome accession found)
[6/30] Fetching HCX9557601 ... ERROR (; No genome accession found)
[7/30] Fetching HDT0929104 ... ERROR (; No genome accession found)
[8/30] Fetching HFV8724395 ... ERROR (; No genome accession found)
[9/30] Fetching HXE6260377 ... ERROR (EFETCH/parse failed: HTTP Error 400: Bad Request)
[10/30] Fetching HXK9204701 ... ERROR (EFETCH/parse failed: HTTP Error 400: Bad Request)
[11/30] Fetching HXX1443132 ... ERROR (EFETCH/parse failed: HTTP Error 400: Bad Request)
[12/30] Fetching HXX1465758 ... ERROR (EFETCH/parse failed: HTTP Error 400: Bad Request)
[13/30] Fetching MCS1346834 ... ERROR (; No genome accession found)
